# Threshold and class-imbalance experiments

The 2022 calibration fold trains on 2016–2021. For each scored year, threshold selection pools out-of-time predictions made before that year. The outer model trains through the previous year. The grid is 0.02–0.98 in steps of 0.01; the score is Macro-F1, with ties resolved towards 0.50. The D0 majority rule remains fixed.

The annual windows hold calendar days together and freeze inspection history at their start. B1 uses inverse-frequency training weights, B2 randomly samples non-Fail records to 1:1, B3 uses train-only SMOTENC for mixed numerical and nominal features, and R0/R1 compare plain and weighted Random Forest. B1–B3 and R0–R1 reuse M4 features. The 2026 holdout is outside this rerun.

In [1]:
import gzip
import json

import pandas as pd

with gzip.open("lab8_results.json.gz", "rt", encoding="utf-8") as file:
    report = json.load(file)
rows = pd.DataFrame([
    dict(code=f["code"], year=f["fold"], threshold=f["threshold"],
         macro_f1=f["subgroups"]["all"]["macro_f1"],
         fail_precision=f["subgroups"]["all"]["fail_precision"],
         fail_recall=f["subgroups"]["all"]["fail_recall"],
         false_positives=f["subgroups"]["all"]["fp"],
         fail_share=f["subgroups"]["all"]["fail_rate"],
         macro_f1_at_half=f["default_metrics"]["macro_f1"],
         recall_at_half=f["default_metrics"]["fail_recall"])
    for f in report["folds"]
])
(rows.groupby("code")
 .mean(numeric_only=True)
 .sort_values("macro_f1", ascending=False)
 .round(4))

      threshold  macro_f1  fail_precision  fail_recall  false_positives  fail_share  macro_f1_at_half  recall_at_half
code                                                                                                                 
M4       0.2967    0.5844          0.3508       0.3829        2534.6667      0.2259            0.4748          0.0424
M12      0.2933    0.5841          0.3483       0.3945        2646.3333      0.2259            0.4745          0.0420
B1       0.5867    0.5840          0.3541       0.3792        2491.0000      0.2259            0.5302          0.7069
M9       0.2967    0.5838          0.3538       0.3861        2554.6667      0.2259            0.4742          0.0418
B2       0.5800    0.5835          0.3464       0.3942        2658.3333      0.2259            0.5275          0.7092
M8       0.2933    0.5820          0.3457       0.3886        2636.3333      0.2259            0.4736          0.0408
R0       0.3000    0.5814          0.3491       0.3642  

In [2]:
focus = ["M4", "B1", "B2", "B3", "R0", "R1"]
rows[rows.code.isin(focus)].sort_values(["year", "code"]).round(4)

   code  year  threshold  macro_f1  fail_precision  fail_recall  false_positives  fail_share  macro_f1_at_half  recall_at_half
20   B1  2023       0.60    0.5791          0.3898       0.3124             1875      0.2488            0.5391          0.7025
21   B2  2023       0.58    0.5834          0.3741       0.3739             2399      0.2488            0.5376          0.7027
22   B3  2023       0.52    0.5760          0.3467       0.5066             3662      0.2488            0.5715          0.5565
5    M4  2023       0.30    0.5852          0.3819       0.3609             2240      0.2488            0.4665          0.0409
23   R0  2023       0.30    0.5805          0.3742       0.3549             2276      0.2488            0.4350          0.0063
24   R1  2023       0.58    0.5823          0.3732       0.3700             2383      0.2488            0.5434          0.6960
45   B1  2024       0.58    0.5835          0.3344       0.4131             2862      0.2176            0.5246 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, layout="constrained")
for ax, (year, points) in zip(axes, sorted(report["curves"]["M4"].items())):
    curve = pd.DataFrame.from_dict(points, orient="index").astype(float)
    curve.index = curve.index.astype(float)
    curve = curve.sort_index()
    curve.plot(ax=ax, lw=2)
    number = rows.loc[(rows.code == "M4") & (rows.year == year), "threshold"].iloc[0]
    ax.axvline(number, color="black", linestyle="--", label=f"Selected: {number:.2f}")
    ax.axvline(.5, color="gray", linestyle=":", label="Default: 0.50")
    ax.set(title=f"Earlier predictions for {year}", xlabel="Fail threshold", ylabel="Score")
    ax.legend(fontsize=8)
plt.show()